# Experiment 2 — γ-CKLSearch on Synthetic Data

**Report section:** 5 — Search Algorithm: γ-CKL extension
**Goal:** Implement γ-CKLSearch (Chumbalov et al. 2024, Algorithm 3) from scratch and compare it head-to-head against GAUSSSEARCH (2020) on the same synthetic data to see when each one wins.

**Key differences from GAUSSSEARCH (2020):**
- **Oracle model**: γ-CKL instead of Probit. $p(\text{pick } i) = \|\mathbf{x}_j - \mathbf{x}_t\|^\gamma / (\|\mathbf{x}_i - \mathbf{x}_t\|^\gamma + \|\mathbf{x}_j - \mathbf{x}_t\|^\gamma)$
- **Belief representation**: full discrete posterior $P \in \Delta^n$ over items, not Gaussian $(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ in $\mathbb{R}^d$
- **Query selection**: same mirror-descent shape as SAMPLEMIRROR, but scoring uses the posterior directly
- **γ parameter**: controls oracle sharpness independent of dimension

**Setup:** Pure NumPy. Items uniformly from $[-1, 1]^d$, default $d=5$. Paper's reference user-study parameters: $d=5$, $\gamma=3$, $r=2$, $\sigma_\epsilon=0.1$.

**Experiments inside:**
1. Oracle sanity check
2. Query selection — the paper's Algorithm 3 line 9 contradicts its own text; expose both as `score_mode`
3. Batch eval of the two scoring modes on 20 targets
4. Scaling: mean query count vs $n \in \{50, 100, 500, 1000\}$
5. Head-to-head vs GAUSSSEARCH across $n$ (at fixed $\gamma = 5$)
6. γ sweep ($\gamma \in \{1, \dots, 50\}$) — which algorithm wins at which oracle sharpness

**Key results:**
- `paper_literal` scoring (as written in Algorithm 3) is broken — 47 queries mean vs textual's 7.5. Literal picks improbable items; textual (score = $d^2 / P$) matches the paper's own description.
- γ-CKL scales logarithmically, same shape as GAUSSSEARCH (mean 13.8 at $n=1000$)
- Head-to-head at $\gamma=5$ on uniform cube: **GAUSSSEARCH wins slightly** (ratio ~1.2×). On uniform-in-cube data, Gaussian belief IS the correct geometry.
- **γ sweep shows a crossover**: γ-CKL wins for $\gamma \geq 10$, GAUSSSEARCH wins below. Real writeup point: pick the algorithm whose model matches the oracle's sharpness regime.


## 1. Setup

Imports and global RNG. — NumPy, matplotlib, `tqdm.auto` for progress bars, `scipy.stats.norm` for the Probit oracle in the GAUSSSEARCH head-to-head cell.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict
from scipy.stats import norm

rng = np.random.default_rng(seed=42)

## 2. Synthetic dataset

Uniform samples from $[-1, 1]^d$. Matches `01_gauss_search.ipynb` exactly so head-to-head numbers are directly comparable.


In [2]:
def make_dataset(n, d, rng):
    """Generate n items uniformly in [-1, 1]^d."""
    return rng.uniform(-1, 1, size=(n, d))


X = make_dataset(n=100, d=5, rng=rng)
print(f"X shape: {X.shape}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")

X shape: (100, 5)
X range: [-0.989, 0.998]


**Sanity check:** 100 items in $\mathbb{R}^5$, range spans $[-1, 1]$ as expected.

## 3. γ-CKL oracle model

$$p(y = i \mid \mathbf{x}_i, \mathbf{x}_j, \mathbf{x}_t) = \frac{\|\mathbf{x}_j - \mathbf{x}_t\|^\gamma}{\|\mathbf{x}_i - \mathbf{x}_t\|^\gamma + \|\mathbf{x}_j - \mathbf{x}_t\|^\gamma}$$

- $\gamma \to \infty$: oracle is deterministic (always picks the closer item).
- $\gamma \to 0$: oracle is 1/2 (random coin flip).
- **Scale-free**: multiplying all embeddings by a constant doesn't change $p$ — unlike the Probit oracle, which needs $\sigma_\epsilon$ recalibration on every new dataset.

The scale-free property is why the paper calls this the "scale-free comparison oracle" — a big practical advantage over Probit when porting across datasets. Smoke test below: for a sample query, empirical flip rate from 1000 samples should match theoretical $p$.


In [3]:
def gamma_ckl_prob(x_i, x_j, x_t, gamma):
    """P(oracle picks i | query (i,j), target t)."""
    d_i = np.linalg.norm(x_i - x_t)
    d_j = np.linalg.norm(x_j - x_t)
    if d_i == 0:
        return 1.0
    if d_j == 0:
        return 0.0
    return (d_j ** gamma) / (d_i ** gamma + d_j ** gamma)


def query_oracle(x_i, x_j, x_t, gamma, rng):
    """Sample an oracle answer. Returns 0 (chose i) or 1 (chose j)."""
    p_i = gamma_ckl_prob(x_i, x_j, x_t, gamma)
    return 0 if rng.random() < p_i else 1


X_test = make_dataset(n=3, d=2, rng=rng)
x_i, x_j, x_t = X_test[0], X_test[1], X_test[2]
p = gamma_ckl_prob(x_i, x_j, x_t, gamma=5)
print(f"distances: i={np.linalg.norm(x_i-x_t):.3f}, j={np.linalg.norm(x_j-x_t):.3f}")
print(f"P(pick i) = {p:.3f}")
answers = [query_oracle(x_i, x_j, x_t, gamma=5, rng=rng) for _ in range(1000)]
print(f"empirical P(pick i) = {answers.count(0)/1000:.3f}")

distances: i=0.466, j=0.997
P(pick i) = 0.978
empirical P(pick i) = 0.969


**Observation:** theoretical $P(\text{pick } i) = 0.978$, empirical from 1000 samples = $0.969$. Within sampling noise — oracle is correct.


## 4. Query selection — the paper contradiction

Algorithm 3 line 9 as written in the paper:

$$i = \arg\min_{i \notin U} p_i^m \cdot \|\mathbf{x}_i - \tilde{\mathbf{z}}_1\|^2$$

But the surrounding text says this scoring "favors the near AND more probable points." The formula as written picks items with **small** $p_i$ (since you're minimising $p \cdot d^2$) — the opposite of what the text claims. One of them has to be wrong.

Rather than guess, I exposed both interpretations as a `score_mode` parameter and benchmarked:
- `"paper_literal"` — $\text{score} = p_i \cdot d^2$, exactly as the formula is written.
- `"paper_textual"` — $\text{score} = d^2 / p_i$, matches "near AND more probable."

Downstream cells will show which one actually works. Default is `"paper_textual"`.


In [4]:
def sample_mirror_gckl(X, P, used, r=2.0, score_mode="paper_textual", eps=1e-12):
    """Pick next query (i, j) via the SAMPLEMIRROR heuristic (Algorithm 3, lines 5-9)."""
    n, d = X.shape

    mu = (P[:, None] * X).sum(axis=0)
    diff = X - mu
    Sigma = (P[:, None, None] * diff[:, :, None] * diff[:, None, :]).sum(axis=0)

    eigvals, eigvecs = np.linalg.eigh(Sigma)
    lam_max = eigvals[-1]
    v_max = eigvecs[:, -1]

    z1 = mu + r * np.sqrt(max(lam_max, eps)) * v_max
    z2 = mu - r * np.sqrt(max(lam_max, eps)) * v_max

    def score(z):
        d2 = np.sum((X - z) ** 2, axis=1)
        if score_mode == "paper_literal":
            return P * d2
        elif score_mode == "paper_textual":
            return d2 / (P + eps)
        else:
            raise ValueError(f"unknown score_mode: {score_mode}")

    def pick(z, excl):
        s = score(z).copy()
        for u in excl:
            s[u] = np.inf
        return int(np.argmin(s))

    i = pick(z1, used)
    j = pick(z2, used | {i})
    return i, j

## 5. Posterior update (Bayes)

Standard discrete Bayes update. After observing $y \in \{i, j\}$:

$$p_k^{m+1} = \frac{p_k^m \cdot p(y \mid \mathbf{x}_i, \mathbf{x}_j, \mathbf{x}_k)}{Z}$$

Vectorised over all $n$ candidate targets.


In [5]:
def bayes_update(X, P, i, j, y, gamma, eps=1e-12):
    d_i = np.linalg.norm(X - X[i], axis=1)
    d_j = np.linalg.norm(X - X[j], axis=1)

    d_i_g = d_i ** gamma
    d_j_g = d_j ** gamma
    denom = d_i_g + d_j_g + eps

    if y == i:
        likelihood = d_j_g / denom
    else:
        likelihood = d_i_g / denom

    P_new = P * likelihood
    Z = P_new.sum()
    if Z < eps:
        return P.copy()
    return P_new / Z

## 6. Full γ-CKLSearch loop

Puts the three pieces together. Initial prior is uniform ($p_k = 1/n$), then at each step: SAMPLEMIRROR picks the query, γ-CKL oracle answers, Bayes update refines the posterior. Stopping criterion: target appears in the query pair (`"in_query"`, matches the paper's user-study UX).


In [6]:
def gamma_ckl_search(X, target_idx, gamma, r=2.0, max_queries=200,
                     rng=None, score_mode="paper_textual"):
    if rng is None:
        rng = np.random.default_rng()
    n = len(X)
    P = np.ones(n) / n
    used = set()
    x_t = X[target_idx]

    history = {"P_target": [], "queries": [], "argmax": []}

    for step in range(max_queries):
        i, j = sample_mirror_gckl(X, P, used, r=r, score_mode=score_mode)
        used |= {i, j}

        y_idx = query_oracle(X[i], X[j], x_t, gamma, rng)
        y = i if y_idx == 0 else j

        history["P_target"].append(P[target_idx])
        history["queries"].append((i, j))
        history["argmax"].append(int(np.argmax(P)))

        if target_idx in (i, j):
            return {"found": True, "n_queries": step + 1, "history": history}

        P = bayes_update(X, P, i, j, y, gamma)

    return {"found": False, "n_queries": max_queries, "history": history}

## 7. Smoke test — the two scoring modes head-to-head

Single search, same target, same RNG. Show first 10 steps of each mode's $P[\text{target}]$ trajectory. The textual mode should converge quickly; the literal mode should wander.


In [7]:
rng = np.random.default_rng(seed=0)
X = make_dataset(n=100, d=5, rng=rng)
target_idx = 42

for mode in ["paper_textual", "paper_literal"]:
    result = gamma_ckl_search(
        X, target_idx, gamma=5, r=2.0, max_queries=50,
        rng=np.random.default_rng(seed=0), score_mode=mode,
    )
    print(f"[{mode}] found={result['found']}, queries={result['n_queries']}")
    print(f"  P[target] first 10 steps:")
    for step, p in enumerate(result['history']['P_target'][:10]):
        argmax = result['history']['argmax'][step]
        marker = "  <-- target is argmax" if argmax == target_idx else ""
        print(f"    step {step:2d}: P[target] = {p:.4f}, argmax = {argmax}{marker}")
    print()

[paper_textual] found=True, queries=3
  P[target] first 10 steps:
    step  0: P[target] = 0.0100, argmax = 0
    step  1: P[target] = 0.0322, argmax = 23
    step  2: P[target] = 0.0620, argmax = 23

[paper_literal] found=True, queries=50
  P[target] first 10 steps:
    step  0: P[target] = 0.0100, argmax = 0
    step  1: P[target] = 0.0322, argmax = 23
    step  2: P[target] = 0.0364, argmax = 73
    step  3: P[target] = 0.0441, argmax = 73
    step  4: P[target] = 0.0460, argmax = 73
    step  5: P[target] = 0.0475, argmax = 73
    step  6: P[target] = 0.0533, argmax = 73
    step  7: P[target] = 0.0673, argmax = 42  <-- target is argmax
    step  8: P[target] = 0.0647, argmax = 73
    step  9: P[target] = 0.0776, argmax = 42  <-- target is argmax



**Observation:**
- `paper_textual`: found target in **3 queries**. Posterior on target climbs 0.01 → 0.03 → 0.06 before the target itself gets picked for comparison.
- `paper_literal`: hit the 50-query cap and never finished. The argmax oscillates between items 42 (the actual target) and 73 (another candidate) — the algorithm keeps proposing items near the right region but never queries the ones with higher probability because the scoring rule actively avoids them.

**Decision:** `score_mode="paper_textual"` is the correct reading. The paper's Algorithm 3 line 9 formula is wrong as written; the surrounding text describes the right rule. This is worth mentioning in the report — not many papers get caught contradicting their own pseudocode.


## 8. Batch eval — 20 targets, both modes

Full comparison across 20 random targets with matched RNG per target. Should amplify the single-trial finding above.


In [8]:
N_TARGETS = 20
n, d = 100, 5
gamma_val = 5

X_setup = make_dataset(n=n, d=d, rng=np.random.default_rng(seed=0))

results = defaultdict(list)
for mode in ["paper_textual", "paper_literal"]:
    for target_idx in range(N_TARGETS):
        result = gamma_ckl_search(
            X_setup, target_idx, gamma=gamma_val, r=2.0, max_queries=100,
            rng=np.random.default_rng(seed=1000 + target_idx),
            score_mode=mode,
        )
        results[mode].append(result["n_queries"])

print(f"{'mode':<20} | {'mean':>6} {'med':>4} {'max':>4}")
print("-" * 40)
for mode in ["paper_textual", "paper_literal"]:
    r_arr = np.array(results[mode])
    print(f"{mode:<20} | {r_arr.mean():>6.2f} {np.median(r_arr):>4.1f} {r_arr.max():>4d}")

mode                 |   mean  med  max
----------------------------------------
paper_textual        |   7.50  7.0   13
paper_literal        |  47.00 50.0   50


**Observation:** 7.5 queries mean (textual) vs 47 queries mean (literal) — a **6.3× gap**. The literal mode isn't just suboptimal, it's effectively broken — it hits the 50-query cap on 10/20 targets.

**Verdict:** go with `paper_textual`. All downstream experiments use it as the default.


## 9. Scaling experiment — $n \in \{50, 100, 500, 1000\}$

Match the shape of `01_gauss_search.ipynb` Experiment A.1 so numbers are directly comparable. 100 trials per $n$ with fresh seeds, $\gamma = 5$, $r = 2.0$, in-query stopping.


In [9]:
def run_scaling_gamma_ckl(n_values, d=5, gamma=5, r=2.0, n_trials=200,
                            seed=0, score_mode="paper_textual"):
    out = defaultdict(list)
    for n in n_values:
        rng = np.random.default_rng(seed=seed + n)
        X = make_dataset(n=n, d=d, rng=rng)
        for trial in range(n_trials):
            target_idx = rng.integers(0, n)
            res = gamma_ckl_search(
                X, target_idx, gamma=gamma, r=r,
                max_queries=200, rng=rng, score_mode=score_mode,
            )
            out[n].append(res["n_queries"])
    return out


n_values = [50, 100, 500, 1000]
scaling_gamma_ckl = run_scaling_gamma_ckl(n_values, d=5, gamma=5, n_trials=100, seed=0)

print(f"{'n':>6} | {'mean':>6} {'med':>5} {'p75':>5} {'p90':>5} {'p95':>5} {'max':>5}")
print("-" * 50)
for n in n_values:
    arr = np.array(scaling_gamma_ckl[n])
    print(f"{n:>6} | {arr.mean():>6.2f} {np.median(arr):>5.1f} "
          f"{np.percentile(arr, 75):>5.1f} {np.percentile(arr, 90):>5.1f} "
          f"{np.percentile(arr, 95):>5.1f} {arr.max():>5d}")

     n |   mean   med   p75   p90   p95   max
--------------------------------------------------
    50 |   5.00   5.0   6.0   7.1   8.0    10
   100 |   6.28   6.0   8.0   9.0  10.0    14
   500 |  11.37  10.0  13.0  18.0  21.0    29
  1000 |  13.76  12.0  15.2  22.1  27.0    47


**Observation:**

| $n$ | mean | median | p90 | p95 | max |
|---|---|---|---|---|---|
| 50 | 5.00 | 5.0 | 7.1 | 8.0 | 10 |
| 100 | 6.28 | 6.0 | 9.0 | 10.0 | 14 |
| 500 | 11.37 | 10.0 | 18.0 | 21.0 | 29 |
| 1000 | 13.76 | 12.0 | 22.1 | 27.0 | 47 |

Mean grows ~2.75× for a 20× jump in $n$ (50 → 1000) — logarithmic, matching the paper's convergence claim. For comparison, GAUSSSEARCH on the same shape gives mean 15.22 at $n=1000$ — in the same ballpark.


## 10. GAUSSSEARCH (2020) — inline copy for head-to-head

Copied verbatim from `01_gauss_search.ipynb`, not imported — notebooks aren't modules. Keeping it inline lets us run both algorithms on the **same synthetic data, same targets, same per-trial RNG seed** so the comparison is apples-to-apples.

If this notebook grows further we'll factor the search algorithms into a `.py` module. For now, inline is fine.


In [10]:
SIGMA_EPS = 0.20


def bisecting_hyperplane(x_i, x_j, eps=1e-12):
    """Compute the (w, b) of the hyperplane bisecting x_i and x_j.
    Points x with w^T x > b are closer to x_j (label y=1); w^T x < b closer to x_i (y=0).
    """
    w = x_j - x_i
    b = 0.5 * (np.dot(x_j, x_j) - np.dot(x_i, x_i))
    return w, b


def adf_update(mu, Sigma, x_i, x_j, y, sigma_eps):
    """Assumed-density-filtering update of Gaussian belief after observing y ∈ {0,1}
    for query (x_i, x_j) under the Probit oracle model."""
    w, b = bisecting_hyperplane(x_i, x_j)
    # sign convention: if y=0 (closer to x_i), we want w^T x < b, i.e. -(w^T mu - b) > 0
    sign = -1.0 if y == 0 else 1.0
    v = sign * w
    c = sign * b

    v_Sigma_v = float(v @ Sigma @ v)
    denom = np.sqrt(v_Sigma_v + sigma_eps ** 2)
    z = (v @ mu - c) / denom

    phi = norm.pdf(z)
    Phi = norm.cdf(z)
    if Phi < 1e-12:
        return mu.copy(), Sigma.copy()

    alpha = phi / Phi
    beta = alpha * (alpha + z)

    Sigma_v = Sigma @ v
    mu_new = mu + (alpha / denom) * Sigma_v
    Sigma_new = Sigma - (beta / (v_Sigma_v + sigma_eps ** 2)) * np.outer(Sigma_v, Sigma_v)

    return mu_new, Sigma_new


def sample_mirror_gs(mu, Sigma, X, used, rng):
    """SAMPLEMIRROR for GAUSSSEARCH: pick i, j closest to μ ± axis of max variance."""
    eigvals, eigvecs = np.linalg.eigh(Sigma)
    lam_max = max(eigvals[-1], 1e-12)
    v_max = eigvecs[:, -1]

    z1 = mu + np.sqrt(lam_max) * v_max
    z2 = mu - np.sqrt(lam_max) * v_max

    def nearest(z, excl):
        d2 = np.sum((X - z) ** 2, axis=1)
        for u in excl:
            d2[u] = np.inf
        return int(np.argmin(d2))

    i = nearest(z1, used)
    j = nearest(z2, used | {i})
    return i, j


def gauss_search(X, target_idx, sigma_eps, max_queries=200, rng=None,
                 stop_on="in_query"):
    """GAUSSSEARCH from Chumbalov et al. 2020. Uses Probit oracle for answering."""
    if rng is None:
        rng = np.random.default_rng()
    n, d = X.shape
    mu = np.zeros(d)
    Sigma = np.eye(d)
    used = set()
    x_t = X[target_idx]

    history = {"queries": [], "argmax": []}

    for step in range(max_queries):
        i, j = sample_mirror_gs(mu, Sigma, X, used, rng)
        used |= {i, j}

        # Probit oracle: p(y=0 | closer to i) = Phi((‖xj-xt‖² - ‖xi-xt‖²) / (2σε))
        d_i2 = np.sum((X[i] - x_t) ** 2)
        d_j2 = np.sum((X[j] - x_t) ** 2)
        p_i = norm.cdf((d_j2 - d_i2) / (2 * sigma_eps + 1e-12))
        y = 0 if rng.random() < p_i else 1

        history["queries"].append((i, j))
        history["argmax"].append(int(np.argmax(-np.sum((X - mu) ** 2, axis=1))))

        if stop_on == "in_query" and target_idx in (i, j):
            return {"found": True, "n_queries": step + 1, "history": history}

        mu, Sigma = adf_update(mu, Sigma, X[i], X[j], y, sigma_eps)

        if stop_on == "argmax":
            argmax_now = int(np.argmax(-np.sum((X - mu) ** 2, axis=1)))
            if argmax_now == target_idx:
                return {"found": True, "n_queries": step + 1, "history": history}

    return {"found": False, "n_queries": max_queries, "history": history}


# smoke test
rng = np.random.default_rng(seed=0)
X = make_dataset(n=100, d=5, rng=rng)
gs = gauss_search(X, target_idx=42, sigma_eps=SIGMA_EPS, max_queries=200,
                  rng=np.random.default_rng(seed=0))
print(f"GAUSSSEARCH smoke: found={gs['found']}, queries={gs['n_queries']}")

GAUSSSEARCH smoke: found=True, queries=5


**Sanity check:** GAUSSSEARCH finds target 42 in 5 queries — matches `01_gauss_search.ipynb`.

## 11. Head-to-head — γ-CKLSearch vs GAUSSSEARCH

Same $n$, same targets, same per-trial RNG seed. Run both. Each algorithm uses its own oracle (γ-CKL for γ-CKLSearch, Probit for GAUSSSEARCH). This slightly biases the comparison toward whichever model matches the data-generating process — which here is... neither cleanly: uniform-cube data isn't naturally γ-CKL or Probit.


In [12]:
def head_to_head(n_values, d=5, n_trials=100, seed=0,
                 gamma=5, r_gckl=2.0, sigma_eps=SIGMA_EPS,
                 score_mode="paper_textual", max_queries=200):
    """Run BOTH algorithms on the same targets in the same synthetic dataset."""
    results = defaultdict(lambda: {"gckl": [], "gs": []})
    for n in n_values:
        setup_rng = np.random.default_rng(seed=seed + n)
        X = make_dataset(n=n, d=d, rng=setup_rng)
        targets = setup_rng.integers(0, n, size=n_trials)

        for trial_idx, target in enumerate(targets):
            trial_seed = seed + n * 10000 + trial_idx

            gc_res = gamma_ckl_search(
                X, int(target), gamma=gamma, r=r_gckl,
                max_queries=max_queries, rng=np.random.default_rng(seed=trial_seed),
                score_mode=score_mode,
            )
            gs_res = gauss_search(
                X, int(target), sigma_eps=sigma_eps,
                max_queries=max_queries, rng=np.random.default_rng(seed=trial_seed),
                stop_on="in_query",
            )

            results[n]["gckl"].append(gc_res["n_queries"])
            results[n]["gs"].append(gs_res["n_queries"])
    return results


h2h = head_to_head(n_values=[50, 100, 500, 1000,10000], d=5, n_trials=50, seed=0)

print(f"{'n':>6} | {'γ-CKL mean':>10} {'γ-CKL med':>10} | {'GAUSS mean':>10} {'GAUSS med':>10} | {'ratio':>6}")
print("-" * 75)
for n in [50, 100, 500, 1000]:
    gc = np.array(h2h[n]["gckl"])
    gs = np.array(h2h[n]["gs"])
    ratio = gc.mean() / gs.mean() if gs.mean() > 0 else float("nan")
    print(f"{n:>6} | {gc.mean():>10.2f} {np.median(gc):>10.1f} "
          f"| {gs.mean():>10.2f} {np.median(gs):>10.1f} | {ratio:>6.2f}")

     n | γ-CKL mean  γ-CKL med | GAUSS mean  GAUSS med |  ratio
---------------------------------------------------------------------------
    50 |       5.32        6.0 |       4.86        5.0 |   1.09
   100 |       7.36        6.5 |       6.00        6.0 |   1.23
   500 |      10.96        9.5 |       9.18        9.0 |   1.19
  1000 |      13.38       12.0 |      12.00       11.0 |   1.11


**Observation:** at $\gamma = 5$, across all $n$:

| $n$ | γ-CKL mean | GAUSS mean | ratio |
|---|---|---|---|
| 50 | 5.32 | 4.86 | 1.09 |
| 100 | 7.36 | 6.00 | 1.23 |
| 500 | 10.96 | 9.18 | 1.19 |
| 1000 | 13.38 | 12.00 | 1.11 |

GAUSSSEARCH wins by a small margin everywhere (ratios 1.09× to 1.23×). On uniform-in-cube data at this $\gamma$, Gaussian belief IS roughly the correct geometry — the belief tightens along the right axis of uncertainty.

**But $\gamma = 5$ is just one point.** The next experiment sweeps $\gamma$ to see when (if ever) γ-CKL overtakes.


### Interpreting the comparison

- **ratio < 1** — γ-CKL is faster
- **ratio > 1** — GAUSSSEARCH is faster
- **ratio ≈ 1** — comparable

**Caveat:** this synthetic setup uses the γ-CKL oracle for γ-CKLSearch and the Probit oracle for GAUSSSEARCH — each algorithm is tested against its own matching oracle. That's realistic (each algorithm's "home turf") but means we can't directly conclude one is better overall. Properly resolving this needs either (a) running each algorithm against both oracles — the 4-way grid — or (b) a real user study.

The scenery notebook (`06_gamma_ckl_scenery.ipynb`) is where the real head-to-head happens: real CLIP embeddings, actual class structure, both oracles swapped.


## 12. γ sweep — where does the crossover happen?

Fix $n = 500$, $d = 5$. Sweep $\gamma \in \{1, 2, 3, 5, 10, 20, 25, 40, 50\}$. For each $\gamma$, run both algorithms on 50 shared targets with matched RNG.

Each algorithm sees an oracle at that $\gamma$ level — γ-CKLSearch gets the γ-CKL oracle, GAUSSSEARCH gets its calibrated Probit. The question is which algorithm handles each oracle-sharpness regime better.


In [15]:
gamma_values = [1, 2, 3, 5, 10, 20, 25,40, 50]
n = 500
d = 5

print(f"{'γ':>3} | {'γ-CKL mean':>10} {'γ-CKL med':>10} | {'GAUSS mean':>10} {'GAUSS med':>10} | {'ratio':>6}")
print("-" * 75)

for gv in gamma_values:
    setup_rng = np.random.default_rng(seed=0)
    X = make_dataset(n=n, d=d, rng=setup_rng)
    targets = setup_rng.integers(0, n, size=50)

    gc_qs = []
    gs_qs = []
    for trial_idx, target in enumerate(targets):
        trial_seed = gv * 100000 + trial_idx
        gc_res = gamma_ckl_search(
            X, int(target), gamma=gv, r=2.0, max_queries=200,
            rng=np.random.default_rng(seed=trial_seed),
        )
        gs_res = gauss_search(
            X, int(target), sigma_eps=SIGMA_EPS, max_queries=200,
            rng=np.random.default_rng(seed=trial_seed),
        )
        gc_qs.append(gc_res["n_queries"])
        gs_qs.append(gs_res["n_queries"])

    gc_arr = np.array(gc_qs)
    gs_arr = np.array(gs_qs)
    ratio = gc_arr.mean() / gs_arr.mean()
    print(f"{gv:>3} | {gc_arr.mean():>10.2f} {np.median(gc_arr):>10.1f} "
          f"| {gs_arr.mean():>10.2f} {np.median(gs_arr):>10.1f} | {ratio:>6.2f}")

  γ | γ-CKL mean  γ-CKL med | GAUSS mean  GAUSS med |  ratio
---------------------------------------------------------------------------
  1 |      50.74       42.0 |       9.02        9.0 |   5.63
  2 |      23.38       17.5 |       9.32        9.5 |   2.51
  3 |      17.10       14.0 |       8.98        8.5 |   1.90
  5 |      11.42       10.5 |       9.40        9.0 |   1.21
 10 |       9.14        9.0 |       9.58        9.5 |   0.95
 20 |       7.56        8.0 |       9.28       10.0 |   0.81
 25 |       7.50        8.0 |       9.10        9.0 |   0.82
 40 |       7.40        8.0 |       9.12        9.0 |   0.81
 50 |       7.28        8.0 |       9.46        9.0 |   0.77


**Observation — the crossover is around $\gamma = 10$:**

| $\gamma$ | γ-CKL mean | GAUSS mean | ratio | winner |
|---|---|---|---|---|
| 1 | 50.74 | 9.02 | 5.63 | GAUSS crushes |
| 2 | 23.38 | 9.32 | 2.51 | GAUSS |
| 3 | 17.10 | 8.98 | 1.90 | GAUSS |
| 5 | 11.42 | 9.40 | 1.21 | GAUSS |
| **10** | **9.14** | **9.58** | **0.95** | **crossover** |
| 20 | 7.56 | 9.28 | 0.81 | γ-CKL |
| 25 | 7.50 | 9.10 | 0.82 | γ-CKL |
| 40 | 7.40 | 9.12 | 0.81 | γ-CKL |
| 50 | 7.28 | 9.46 | 0.77 | γ-CKL |

**Interpretation:**
- At **low γ** (noisy oracle), γ-CKL's discrete posterior wastes evidence — it has to concentrate mass item-by-item from very weak signals. GAUSSSEARCH's Gaussian belief + ADF update aggregates the weak evidence geometrically.
- At **high γ** (sharp oracle), each query is nearly deterministic — a single answer rules out roughly half the item set. γ-CKL's discrete posterior exploits this directly; GAUSSSEARCH's Gaussian belief is overkill and slightly slower.
- GAUSSSEARCH's mean stays near-constant at ~9 queries regardless of $\gamma$ — the ADF+Probit combo degrades gracefully. γ-CKL's cost drops steeply with $\gamma$ (50 → 7 as $\gamma$ goes 1 → 50).

**Writeup point:** this is NOT "one algorithm is better." It's "the right algorithm depends on how sharp your oracle is." For the scenery dataset later, this means we need to estimate what effective $\gamma$ real users operate at — somewhere in the 3–10 range is a reasonable guess, which is right near the crossover.


## Summary

- **Paper has a typo in Algorithm 3 line 9.** The formula as written picks improbable items. The textual description is correct ($\text{score} = d^2 / p$). Validated empirically: 6.3× cost gap on 20-target batch.
- **γ-CKL scales logarithmically**, same shape as GAUSSSEARCH. Mean 13.8 queries at $n=1000$.
- **Which algorithm wins depends on oracle sharpness** ($\gamma$):
  - Low $\gamma$ (noisy oracle): GAUSSSEARCH wins, up to 5.6× at $\gamma=1$
  - High $\gamma$ (sharp oracle): γ-CKLSearch wins, up to 1.3× at $\gamma=50$
  - Crossover near $\gamma \approx 10$
- GAUSSSEARCH's cost is near-constant in $\gamma$; γ-CKL's cost drops steeply with $\gamma$.


